In [ ]:
import scanpy as sc
import os 
import sys

from omegaconf import OmegaConf as om
from tahoe_x1.utils.util import compute_lisi_scores

sys.path.insert(0, os.path.abspath('..'))
from scripts.inference.predict_embeddings import predict_embeddings

In [ ]:
model_name = "Tx1-70m"
cfg = {
  "model_name": model_name,
  "paths": {
    "hf_repo_id": "tahoebio/Tahoe-x1",
    "hf_model_size": "70m",
    "adata_input":  "../kim_test.h5ad",
  },
  "data": {
    "cell_type_key": "cell_type",  #"cell_line"  #"OncotreeLineage"
    "gene_id_key": "ensembl_id"  #"feature_id"
  },
  "predict": {
    "seq_len_dataset": 2048,  #8192
    "return_gene_embeddings": False,  # Whether to extract gene embeddings
    "use_chem_inf": False,  # Whether to use chemical information for prediction if the model is trained with chemical information
  },
  "plot":
  {
    "save_dir": "./figures"
  }
}

cfg = om.create(cfg)

In [ ]:
cfg

In [ ]:
# Extract embeddings
adata = predict_embeddings(cfg)

In [ ]:
cell_array = adata.obsm[model_name]

In [ ]:
# Calculate LISI
cell_type_key = cfg.data.cell_type_key
lisi_score = compute_lisi_scores(cell_array, 
                                adata.obs[cell_type_key].values, 
                                20) 
print(f"LISI score: {lisi_score:.4f}")


# Plotting
adata.obs[cell_type_key] = adata.obs[cell_type_key].astype('category')
sc.pp.neighbors(adata, use_rep=model_name)
sc.tl.umap(adata)
fig = sc.pl.umap(adata, 
           color=[cell_type_key], 
           frameon=False, 
           wspace=0.4, 
           title=[f"{model_name} LISI:{lisi_score:.2f}"],
           return_fig=True)

# Save figure
save_dir = cfg.plot.save_dir
os.makedirs(save_dir, exist_ok=True)
fig.savefig(f"{save_dir}/{model_name}_{cfg.paths.adata_input.split('/')[-1].replace('.h5ad','')}.png", dpi=300, bbox_inches="tight")

In [ ]:
#PCA

sc.pp.normalize_total(adata, target_sum=10000, inplace=True)
sc.pp.log1p(adata)
sc.tl.pca(adata, n_comps=15)
sc.pp.neighbors(adata, use_rep="X_pca")
sc.tl.umap(adata)

lisi_score = compute_lisi_scores(adata.obsm["X_pca"], adata.obs[cell_type_key], 20)
fig = sc.pl.umap(adata, color=cell_type_key, title=f"PCA LISI:{lisi_score:.2f}", return_fig=True, frameon=False,)

fig.savefig(f"{save_dir}/PCA_{cfg.paths.adata_input.split('/')[-1].replace('.h5ad','')}.png", dpi=300, bbox_inches="tight")
